# TrashTrack — Training Pipeline (Colab)
**MSDS 498 Capstone · Group 4**

This notebook runs the full training + evaluation pipeline on a free Colab T4 GPU.

**Datasets (already converted & uploaded to Drive):**
- TACO — 1,200 train / 300 val (street-level, 60 classes → single "litter")
- UAVVaste — 618 train / 154 val (drone/aerial)
- Glasgow — 255 train / 63 val (street-level at distance, RoLID-11K stand-in)
- Merged — 2,073 train (all three combined)

**Before you start:** `Runtime → Change runtime type → T4 GPU`


## 1. Setup

In [ ]:
# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Clone the repo
!git clone https://github.com/Nandini1Bag/trashtrack.git
%cd trashtrack

In [ ]:
# Install dependencies
!pip install -q ultralytics pillow numpy pydantic

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/trashtrack_data'
!mkdir -p {DRIVE_BASE}/weights
!mkdir -p {DRIVE_BASE}/results
print(f"Drive storage at: {DRIVE_BASE}")

## 2. Load Datasets from Drive
Datasets were converted locally and uploaded as a zip. No downloading needed.

In [ ]:
# Unzip datasets from Drive
!unzip -q {DRIVE_BASE}/datasets.zip -d /content/trashtrack/
print("Unzipped!")

## 3. Verify Datasets

In [ ]:
import os

def count_files(path, ext):
    if not os.path.exists(path):
        return 0
    return len([f for f in os.listdir(path) if f.endswith(ext)])

print("Dataset Summary:")
print("=" * 55)
total_train = 0
for ds in ["taco", "uavvaste", "rolid11k", "merged"]:
    for split in ["train", "val"]:
        imgs = count_files(f"datasets/{ds}/images/{split}", ".jpg") + \
               count_files(f"datasets/{ds}/images/{split}", ".png")
        lbls = count_files(f"datasets/{ds}/labels/{split}", ".txt")
        if imgs > 0:
            print(f"  {ds:12s} {split:5s}  {imgs:6d} images  {lbls:6d} labels")
            if split == "train" and ds != "merged":
                total_train += imgs
print("=" * 55)
print(f"  Total training images (excl merged): {total_train}")

# Quick sanity check
assert count_files("datasets/taco/images/train", ".jpg") >= 1200, "TACO train count too low!"
assert count_files("datasets/merged/images/train", ".jpg") + \
       count_files("datasets/merged/images/train", ".png") >= 2000, "Merged count too low!"
print("\n All counts look correct.")

In [ ]:
# Spot-check a label file
import random, glob

label_files = glob.glob("datasets/taco/labels/train/*.txt")
if label_files:
    sample = random.choice(label_files)
    print(f"Sample: {sample}")
    print(open(sample).read())
    print("\nFormat: class_id cx cy w h (normalized)")
    print("class_id 0 = litter (all classes collapsed)")

## 4. Run A — TACO-Only Baseline
Train on TACO alone to establish the single-dataset baseline.

⏱ ~3-5 hours on T4 for 100 epochs.

In [ ]:
!yolo detect train \
    data=configs/taco.yaml \
    model=yolov8s.pt \
    epochs=100 \
    imgsz=640 \
    batch=16 \
    patience=20 \
    name=run_a_taco_only \
    save=True \
    plots=True

In [ ]:
# Evaluate Run A on TACO val
!yolo detect val \
    data=configs/taco.yaml \
    model=runs/detect/run_a_taco_only/weights/best.pt \
    name=run_a_eval_taco \
    plots=True

print("\n🎯 Target: mAP@0.5 ≥ 0.45")

In [ ]:
# Save Run A weights to Drive
!cp runs/detect/run_a_taco_only/weights/best.pt {DRIVE_BASE}/weights/run_a_best.pt
!cp -r runs/detect/run_a_taco_only {DRIVE_BASE}/results/
print("Run A weights + results saved to Drive")

## 5. Run B — Merged Training (All 3 Datasets)
Train on TACO + UAVVaste + Glasgow combined, then evaluate on each separately.

⏱ ~5-8 hours on T4 for 100 epochs.

In [ ]:
!yolo detect train \
    data=configs/merged.yaml \
    model=yolov8s.pt \
    epochs=100 \
    imgsz=640 \
    batch=16 \
    patience=20 \
    name=run_b_merged \
    save=True \
    plots=True

In [ ]:
# Evaluate Run B on each dataset separately (ST-04)
print("=" * 60)
print("  ST-04: Cross-Dataset Generalization Benchmark")
print("=" * 60)

for ds, config, target in [
    ("TACO", "configs/taco.yaml", 0.45),
    ("UAVVaste", "configs/uavvaste.yaml", 0.35),
    ("Glasgow", "configs/rolid11k.yaml", 0.30),
]:
    print(f"\n{'─'*40}")
    print(f"  {ds}  (target mAP@0.5 ≥ {target})")
    print(f"{'─'*40}")

    !yolo detect val \
        data={config} \
        model=runs/detect/run_b_merged/weights/best.pt \
        name=run_b_eval_{ds.lower()} \
        plots=True

In [ ]:
# Save Run B weights to Drive
!cp runs/detect/run_b_merged/weights/best.pt {DRIVE_BASE}/weights/run_b_best.pt
!cp -r runs/detect/run_b_merged {DRIVE_BASE}/results/
print("Run B weights + results saved to Drive")

## 6. Compare Results

In [ ]:
import pandas as pd
from pathlib import Path

def read_results(run_dir):
    csv_path = Path(run_dir) / "results.csv"
    if not csv_path.exists():
        return None
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    return df.iloc[-1] if len(df) > 0 else None

results = []

# Run A
r = read_results("runs/detect/run_a_taco_only")
if r is not None:
    results.append({
        "Run": "A (TACO only)",
        "Eval": "TACO",
        "mAP@0.5": round(float(r.get("metrics/mAP50(B)", 0)), 4),
        "mAP@0.5:0.95": round(float(r.get("metrics/mAP50-95(B)", 0)), 4),
        "Precision": round(float(r.get("metrics/precision(B)", 0)), 4),
        "Recall": round(float(r.get("metrics/recall(B)", 0)), 4),
    })

# Run B
r = read_results("runs/detect/run_b_merged")
if r is not None:
    results.append({
        "Run": "B (Merged)",
        "Eval": "All (training)",
        "mAP@0.5": round(float(r.get("metrics/mAP50(B)", 0)), 4),
        "mAP@0.5:0.95": round(float(r.get("metrics/mAP50-95(B)", 0)), 4),
        "Precision": round(float(r.get("metrics/precision(B)", 0)), 4),
        "Recall": round(float(r.get("metrics/recall(B)", 0)), 4),
    })

if results:
    df = pd.DataFrame(results)
    print("\n📊 Results Comparison")
    print("=" * 70)
    print(df.to_string(index=False))

print("\n\n🎯 Success Thresholds:")
print("─" * 40)
for ds, t in [("TACO", 0.45), ("UAVVaste", 0.35), ("Glasgow", 0.30)]:
    print(f"  {ds:12s}  mAP@0.5 ≥ {t}")
print(f"  {'Precision':12s}  @ conf≥0.50 ≥ 0.70")
print(f"  {'Recall':12s}  @ conf≥0.25 ≥ 0.50")

## 7. Save Everything to Drive

In [ ]:
!cp -r runs/detect {DRIVE_BASE}/results/ 2>/dev/null

print("📁 Saved to Drive:")
!ls -la {DRIVE_BASE}/weights/
print()
!ls {DRIVE_BASE}/results/
print("\n✅ All done. You can close Colab now.")

## 8. Use the Trained Model in Your App

Download the best weights from Drive and run locally:

```bash
# Copy weights from Google Drive to your project
cp ~/Google\ Drive/My\ Drive/trashtrack_data/weights/run_b_best.pt weights/best.pt

# Start the API with fine-tuned model
TT_MODEL_WEIGHTS=weights/best.pt uvicorn trashtrack.api:app --port 8000

# Start the React frontend (separate terminal)
cd frontend && npm run dev
```

Open http://localhost:3000, upload a street photo — now it actually detects litter.
